In [1]:
import pandas as pd
import numpy as np
import re

with open("/home/cry_more/ongoing/LogRecall/data/HDFS.log",'r')as f:
    df = [line.strip() for line in f if line.strip()]

X = pd.DataFrame(df,columns=['logs'])
y = pd.read_csv("/home/cry_more/ongoing/LogRecall/data/anomaly_label.csv")

In [14]:
X.head()

,logs,BlockId
0,081109 203518 143 INFO dfs.DataNode$DataXceive...,blk_-1608999687919862906
1,081109 203518 35 INFO dfs.FSNamesystem: BLOCK*...,blk_-1608999687919862906
2,081109 203519 143 INFO dfs.DataNode$DataXceive...,blk_-1608999687919862906
3,081109 203519 145 INFO dfs.DataNode$DataXceive...,blk_-1608999687919862906
4,081109 203519 145 INFO dfs.DataNode$PacketResp...,blk_-1608999687919862906


In [16]:
len(X["logs"].unique())

11173511

In [ ]:
import re
import pandas as pd
import os

def mask_log(raw_log: str) -> str:
    cleaned = re.sub(r'^\s*\d+\s+\d+\s+\d+\s+[A-Z]+\s+[^:]+:\s*', '', raw_log)
    cleaned = re.sub(r'blk_-?\d+', '<BLOCK_ID>', cleaned)
    cleaned = re.sub(r'\b\d{1,3}(?:\.\d{1,3}){3}\b', '<IP>', cleaned)
    cleaned = re.sub(r'\b\d+\b', '<NUM>', cleaned)
    return cleaned.strip()

def get_id(x):
    m = re.search(r"blk_", x)
    if m is None:
        return "unknown"
    if m is None:
        return None
    st=m.start()
    en = x.find(" ",st)
    if en==-1:
        en = len(x)
    return x[st:en] 

def process_in_chunks(input_file, output_file, chunk_size=100000):
    chunk_list = []
    
    with open(input_file, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            raw_line = line.strip()
            blk_id = get_id(raw_line)
            template = mask_log(raw_line)

            chunk_list.append({
                'raw_log':raw_line,
                "template_log":template,
                "block_id":blk_id
            })
            
            if (i + 1) % chunk_size == 0:
                df = pd.DataFrame(chunk_list)
                file_exists = os.path.exists(output_file)
                df.to_csv(output_file, mode='a', header=not file_exists, index=False)
                chunk_list.clear() 
        
        if chunk_list:
            df = pd.DataFrame(chunk_list)
            file_exists = os.path.exists(output_file)
            df.to_csv(output_file, mode='a', header=not file_exists, index=False)

process_in_chunks('/home/cry_more/ongoing/LogRecall/data/HDFS.log', 'HDFS_parsed.csv')

In [ ]:
import pandas as pd
import sqlite3
import time

def build_sqlite_storage(parsed_logs_csv, labels_csv, db_path='logrecall_metadata.db'):
    # 1. Connect to SQLite
    print(f"Connecting to {db_path}...")
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    # 2. Create the exact Two-Table Schema
    print("Creating table schemas...")
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS block_labels (
            blk_id TEXT PRIMARY KEY,
            label TEXT
        )
    ''')
    
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS logs (
            log_id INTEGER PRIMARY KEY AUTOINCREMENT,
            blk_id TEXT,
            template_id INTEGER,
            raw_log TEXT
        )
    ''')
    
    # 3. Load the Labels (The Truth Table)
    print(f"Loading {labels_csv} into 'block_labels' table...")
    start_time = time.time()
    labels_df = pd.read_csv(labels_csv)
    labels_df = labels_df.rename(columns={'BlockId': 'blk_id', 'Label': 'label'})
    # Assuming your CSV has columns 'blk_id' and 'label'
    labels_df.to_sql('block_labels', conn, if_exists='append', index=False)
    print(f"Labels loaded in {time.time() - start_time:.2f} seconds.")

    # 4. Load the 11-Million Logs (The Reality Table)
    print(f"Streaming {parsed_logs_csv} into 'logs' table (Chunking enabled)...")
    start_time = time.time()
    
    # We will assign a unique template_id to each unique template string on the fly
    template_dict = {}
    current_template_id = 1
    
    chunk_size = 100000
    rows_inserted = 0
    
    # The Memory Valve: Read the 2.45GB file 100k lines at a time
    for chunk in pd.read_csv(parsed_logs_csv, chunksize=chunk_size):
        chunk = chunk.rename(columns={'block_id': 'blk_id'})
        records_to_insert = []
        
        for _, row in chunk.iterrows():
            template_text = row['template_log']
            
            # If we haven't seen this template before, give it a new ID
            if template_text not in template_dict:
                template_dict[template_text] = current_template_id
                current_template_id += 1
                
            t_id = template_dict[template_text]
            
            # Prepare the row for the database
            records_to_insert.append((row['blk_id'], t_id, row['raw_log']))
            
        # Bulk insert the chunk into SQLite
        cursor.executemany('''
            INSERT INTO logs (blk_id, template_id, raw_log) 
            VALUES (?, ?, ?)
        ''', records_to_insert)
        
        rows_inserted += len(chunk)
        print(f"Inserted {rows_inserted} physical logs...")
    
    conn.commit()
    print(f"11 Million Logs loaded in {time.time() - start_time:.2f} seconds.")

    # 5. Build Indexes for Microsecond Queries
    print("Building database indexes (This takes a minute)...")
    cursor.execute('CREATE INDEX idx_blk_id ON logs (blk_id);')
    cursor.execute('CREATE INDEX idx_template_id ON logs (template_id);')
    conn.commit()
    
    # 6. Save the Template Mapping for FAISS
    print("Saving Template Dictionary for FAISS generation...")
    templates_df = pd.DataFrame(list(template_dict.items()), columns=['template_log', 'template_id'])
    templates_df.to_csv('template_dictionary.csv', index=False)
    
    conn.close()
    print("\nSQLITE STORAGE ONLINE. Ready for FAISS.")

# Execute it (Update these filenames to match yours)
if __name__ == "__main__":
    build_sqlite_storage('./HDFS_parsed.csv', '/home/cry_more/ongoing/LogRecall/data/anomaly_label.csv')

In [13]:
import pandas as pd
import numpy as np
import faiss
import torch
from sentence_transformers import SentenceTransformer
import time

def build_faiss_index(dict_csv='template_dictionary.csv', 
                      index_path='logrecall_core.index', 
                      custom_model_path='/home/cry_more/ongoing/LogRecall/artifacts/stage1_encoder.pth'):
    
    print(f"Loading conceptual dictionary: {dict_csv}...")
    df = pd.read_csv(dict_csv)
    
    # 1. Load the Base Architecture
    print("Booting base AI architecture (all-MiniLM-L6-v2)...")
    model = SentenceTransformer('sentence-transformers/all-MiniLM-L12-v2') 
    
    # 2. Inject Custom Stage 1 Weights (with prefix stripper)
    print(f"Injecting fine-tuned weights from {custom_model_path}...")
    raw_state_dict = torch.load(custom_model_path, map_location=torch.device('cpu'))
    
    clean_state_dict = {}
    for key, value in raw_state_dict.items():
        new_key = key.replace('model.', '', 1) if key.startswith('model.') else key
        clean_state_dict[new_key] = value
    
    try:
        model[0].auto_model.load_state_dict(clean_state_dict)
        print("Custom AI Model booted and locked.")
    except Exception as e:
        print(f"Injection Failed: {e}")
        return

    # 3. Encode the Unique Concepts
    print(f"Embedding {len(df)} unique templates into 384-dimensional space...")
    start_time = time.time()
    
    templates = df['template_log'].tolist()
    vectors = model.encode(templates, batch_size=256, show_progress_bar=True)
    vectors = np.array(vectors).astype('float32') 
    
    print(f"Math calculation complete in {time.time() - start_time:.2f} seconds.")

    # 4. Construct the Foreign Key Bridge
    print("Constructing FAISS IndexIDMap...")
    dimension = vectors.shape[1] 
    faiss_index = faiss.IndexIDMap(faiss.IndexFlatIP(dimension))
    
    template_ids = df['template_id'].values.astype('int64')
    faiss_index.add_with_ids(vectors, template_ids)
    
    # 5. Lock to Disk
    faiss.write_index(faiss_index, index_path)
    print(f"\nFAISS BRAIN ONLINE. Saved to {index_path}.")
    print(f"Total AI Memory Footprint: {faiss_index.ntotal} concepts mapped.")

if __name__ == "__main__":
    build_faiss_index()

Loading conceptual dictionary: template_dictionary.csv...
Booting base AI architecture (all-MiniLM-L6-v2)...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1306.29it/s]


Injecting fine-tuned weights from /home/cry_more/ongoing/LogRecall/artifacts/stage1_encoder.pth...
Custom AI Model booted and locked.
Embedding 34388 unique templates into 384-dimensional space...


Batches: 100%|██████████| 135/135 [00:37<00:00,  3.64it/s]


Math calculation complete in 37.28 seconds.
Constructing FAISS IndexIDMap...

FAISS BRAIN ONLINE. Saved to logrecall_core.index.
Total AI Memory Footprint: 34388 concepts mapped.


In [ ]:
parsed_log = pd.read_csv("/home/cry_more/ongoing/LogRecall/data/HDFS_parsed.csv")
df = pd.concat([X["BlockId"],parsed_log],axis=True)

template_df = (
    df.groupby("BlockId", sort=False).agg(
        Template=("Template", "\n".join)
    ).reset_index()
)

In [ ]:
df = X.merge(y,on='BlockId',how='inner')
orig_df = (
    df.groupby('BlockId',sort=False).agg(
        logs=('logs','\n'.join),
        label=('Label','first')
    ).reset_index()
)

In [ ]:
template_df.to_parquet("/home/cry_more/ongoing/LogRecall/data/template_df.parquet",index=False)
orig_df.to_parquet("/home/cry_more/ongoing/LogRecall/data/orig_df.parquet",index=False)

ArrowKeyError: No type extension with name arrow.py_extension_type found